# 07 · Modelo entrenado: evaluación en validación y análisis de la búsqueda

**Qué contiene.** Celdas 0–1: carga el adaptador y evalúa el agente sobre validación, sin prompt de sistema. Celda 2: comparación base contra entrenado. Celdas 3–5: diagnósticos de las consultas de búsqueda (exploratorios, sin GPU). Celda 6: experimento con trayectorias v2 (consulta = pregunta sin fechas ni muletillas).

| | |
|---|---|
| **Runtime** | GPU T4 (Entorno de ejecución → Cambiar tipo de entorno → T4 GPU) para las celdas 0–1; el resto, CPU |
| **Entradas (`MyDrive/qlora-series-ar/data/`)** | `qlora_completo/adapter`, `eval_val_final2.csv`, `specs_val_v2.csv`, `baseline_val_tecnica_completa_optimizado.csv`, `trayectorias_train_v1.jsonl` |
| **Salidas** | `finetuned_val_tecnica_completa_basico.csv` y `_episodios.jsonl`; `trayectorias_train_v2.jsonl` (celda 6) |
| **Secretos de Colab** | `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY` |
| **Costo y tiempo** | Sin costo de API. Del orden de la corrida de validación del base (unos 50 minutos) |
| **Orden** | Después de `05` y `06`; antes de `08` |
| **En el repositorio** | `agente.py`, `corrida.py`, `modelo.py`, `scripts/evaluar.py`. Las celdas 2–6 son exploratorias y no tienen módulo |

**Advertencias.**
- Las trayectorias v2 (celda 6) no se usaron: el modelo evaluado se entrenó con v1.
- Este notebook eligió reglas de consulta mirando validación; el conjunto de test no se tocó.


In [ ]:
!pip -q install unsloth langfuse
print("CELDA 1 de 2: PREPARACIÓN (MODELO ENTRENADO, LANGFUSE, ARCHIVOS)")
import os
os.environ["UNSLOTH_DISABLE_CPU_FALLBACK"] = "1"
from unsloth import FastLanguageModel
import gc, json, shutil
import torch
import logging, warnings
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
from google.colab import drive, userdata

assert torch.cuda.is_available(), "No hay GPU: Runtime > Change runtime type > T4 GPU"
drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
ADAPTADOR = f"{RUTA}/qlora_completo/adapter"
assert os.path.exists(f"{ADAPTADOR}/adapter_config.json"), f"No encuentro el adaptador en {ADAPTADOR}"
MAX_SEQ = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTADOR, max_seq_length=MAX_SEQ, load_in_4bit=True)
FastLanguageModel.for_inference(model)
assert model.max_seq_length == MAX_SEQ, f"El modelo quedó con max_seq_length={model.max_seq_length}"
print(f"Modelo listo | max_seq_length real: {model.max_seq_length} | memoria de GPU: "
      f"{torch.cuda.memory_allocated() / 1e9:.1f} GB | tiene adaptador (peft_config): {hasattr(model, 'peft_config')}")

# ---------- Langfuse (cuenta de EE.UU.) ----------
HOST = "https://us.cloud.langfuse.com"
os.environ["LANGFUSE_PUBLIC_KEY"] = userdata.get("LANGFUSE_PUBLIC_KEY").strip()
os.environ["LANGFUSE_SECRET_KEY"] = userdata.get("LANGFUSE_SECRET_KEY").strip()
os.environ["LANGFUSE_HOST"] = HOST
os.environ["LANGFUSE_BASE_URL"] = HOST
from langfuse import get_client
lf = get_client()
print("Langfuse auth_check:", lf.auth_check())

# ---------- Archivos de trabajo (copias locales) ----------
for nombre in ["cache_api.db", "busqueda.db"]:
    if not os.path.exists(f"/content/{nombre}"):
        shutil.copy(f"{RUTA}/{nombre}", f"/content/{nombre}")
tools = json.load(open(f"{RUTA}/tools_v1.json"))
print("Herramientas:", [t["function"]["name"] for t in tools])

In [ ]:
print("CELDA 2 de 2: AGENTE + EVALUADOR + CORRIDA CON LANGFUSE (modelo entrenado)")
assert all(x in globals() for x in ["model", "tokenizer", "lf", "RUTA", "tools"]), "Ejecutá antes la CELDA 1 de 2"
assert model.max_seq_length >= 4096, f"El modelo quedó con max_seq_length={model.max_seq_length}: hay que recargarlo"
import re, json, time, sqlite3, shutil, unicodedata
import logging, warnings
from collections import Counter
import numpy as np
import pandas as pd
import requests
import torch
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)

# ---------- Configuración ----------
SPLIT = "val"
ESTILO = "tecnica"            # "tecnica" o "coloquial"
N_PRUEBA = None               # None = todo el split; un número = prueba chica
PROMPTS = ["basico"]          # el modelo se entrenó sin prompt de sistema
N_MUESTRA_BASICO = None       # None = las 299 completas, para comparar con el baseline
MAX_PASOS = 6
MAX_NUEVOS = 300
SAVE_CACHE = True
ETIQUETA_MODELO = "qwen3-4b-instruct-2507-qlora"
BASE = "https://apis.datos.gob.ar/series/api/series"
HEADERS = {"User-Agent": "Mozilla/5.0"}
ETIQUETA = ("prueba" if N_PRUEBA else "completa") + "_" + "-".join(PROMPTS)

PROMPT_OPT = """Sos un asistente que responde preguntas sobre estadísticas oficiales de Argentina usando tres herramientas. Seguí siempre este procedimiento:
1. Llamá a search_series con 2 a 6 palabras clave del tema (sin fechas ni números). Elegí, entre los candidatos devueltos, la serie cuyo título, distribución y frecuencia (mensual, trimestral o anual) mejor coincidan con lo que se pide. Usá solo ids devueltos por search_series; nunca inventes ids.
2. Llamá a get_series con la serie elegida:
- Valor de un período: start_date y end_date iguales, en formato AAAA (series anuales) o AAAA-MM.
- Último dato disponible: limit=1 y sort="desc", sin fechas.
- Variación interanual: representation_mode="percent_change_a_year_ago", start_date un año antes del período pedido y end_date el período pedido, con fechas completas AAAA-MM-DD (día 01; en series trimestrales, el primer mes del trimestre). La API devuelve una fracción (0.05 equivale a 5%) y la última fila corresponde al período pedido.
3. Llamá a submit_answer con la respuesta, el valor y la fecha del dato (formato AAAA-MM, o AAAA si la serie es anual). Para variaciones, el valor va en porcentaje (la fracción multiplicada por 100).
Si get_series devuelve data vacía, la API no tiene datos de esa serie para ese período: entregá submit_answer con valor y fecha en null y explicalo en la respuesta. No inventes valores."""
SISTEMA = {"basico": None, "optimizado": PROMPT_OPT}

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
PARAMS_TOOL = {"ids", "start_date", "end_date", "representation_mode", "limit", "sort"}
cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def clave(params):
    p = {k: str(v) for k, v in params.items() if v is not None}
    return json.dumps(dict(sorted(p.items())), ensure_ascii=False)

# ---------- Herramientas: caché primero, API real si falta ----------
class Herramientas:
    def __init__(self):
        self.cache = sqlite3.connect("/content/cache_api.db")
        self.fts = sqlite3.connect("/content/busqueda.db")
        self.fallos_cache, self.sin_red = 0, 0

    def search_series(self, query, k=10):
        toks = [t for t in re.findall(r"[a-z0-9]+", norm(query)) if t not in STOP and len(t) > 1]
        toks = list(dict.fromkeys(toks))
        if not toks:
            return "[]"
        partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
        rows = self.fts.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                                "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
        return json.dumps([{"serie_id": s, "titulo": str(cat.loc[s, "serie_titulo"]),
                            "distribucion": str(cat.loc[s, "distribucion_titulo"]),
                            "frecuencia": str(cat.loc[s, "frecuencia"])} for s, _ in rows], ensure_ascii=False)

    def _api(self, args):
        params = json.loads(clave(args))
        for intento in range(3):
            try:
                r = requests.get(BASE, params=params, headers=HEADERS, timeout=30)
                if r.status_code in (200, 400):
                    return r.status_code, r.json()
            except Exception:
                pass
            time.sleep(1 + intento)
        return None, None

    def get_series(self, args):
        extra = set(args) - PARAMS_TOOL
        if extra or "ids" not in args:
            return json.dumps({"status": 400, "error": f"Parámetros no permitidos o faltantes: {sorted(extra) or ['ids']}"},
                              ensure_ascii=False)
        k = clave(args)
        fila = self.cache.execute("SELECT status, cuerpo FROM cache WHERE clave=?", (k,)).fetchone()
        if fila:
            status, js = fila[0], json.loads(fila[1])
        else:
            self.fallos_cache += 1
            status, js = self._api(args)
            if status is None:
                self.sin_red += 1
                return json.dumps({"status": None, "error": "no se pudo consultar la API"}, ensure_ascii=False)
            self.cache.execute("INSERT OR REPLACE INTO cache VALUES (?,?,?,?)",
                               (k, status, json.dumps(js, ensure_ascii=False),
                                time.strftime("%Y-%m-%dT%H:%M:%S+00:00", time.gmtime())))
            self.cache.commit()
        if status != 200:
            return json.dumps({"status": status, "error": js.get("errors", js)}, ensure_ascii=False)
        sid = str(args["ids"])
        c = cat.loc[sid] if sid in cat.index else None
        return json.dumps({"status": 200, "serie_id": sid,
                           "titulo": None if c is None else str(c["serie_titulo"]),
                           "unidades": None if (c is None or pd.isna(c["serie_unidades"])) else c["serie_unidades"],
                           "frecuencia": None if c is None else str(c["frecuencia"]),
                           "data": js.get("data", [])}, ensure_ascii=False)

H = Herramientas()

# ---------- Evaluador (el mismo validado con el agente oráculo) ----------
def periodo(f, freq):
    f = "" if f is None else str(f)
    return f[:4] if freq == "anual" else f[:7]

def valor_fecha(tipo, freq, d, params):
    if tipo == "sin_datos" or not d or d[0][1] is None or d[-1][1] is None:
        return None, None
    if tipo == "ultimo_dato":
        return d[0][1], d[0][0]
    if tipo == "valor_puntual":
        return d[0][1], params["start_date"]
    return round(d[-1][1] * 100, 2), d[-1][0]

def cerca(a, b, tol_abs):
    return abs(a - b) <= max(tol_abs, 1e-3 * abs(b))

def puntuar(tipo, freq, g, sub, ids_ultima, objetivo):
    r = {"termino": sub is not None, "serie_ok": ids_ultima == objetivo}
    if sub is None:
        r.update(valor_ok=False, valor_flex=False, fecha_ok=False)
    elif tipo == "sin_datos":
        nulo = sub.get("valor") is None
        r.update(valor_ok=nulo, valor_flex=nulo, fecha_ok=sub.get("fecha") is None)
    else:
        v = sub.get("valor")
        num = isinstance(v, (int, float)) and not isinstance(v, bool)
        tol = 0.011 if tipo == "variacion_interanual" else 5e-5
        ok_e = bool(num and cerca(v, g["valor"], tol))
        ok_f = ok_e or bool(tipo == "variacion_interanual" and num and cerca(v, g["valor"] / 100, 1e-4))
        r.update(valor_ok=ok_e, valor_flex=ok_f,
                 fecha_ok=sub.get("fecha") is not None and periodo(sub["fecha"], freq) == periodo(g["fecha"], freq))
    r["correcta"] = bool(r["termino"] and r["serie_ok"] and r["valor_ok"] and r["fecha_ok"])
    r["correcta_flex"] = bool(r["termino"] and r["serie_ok"] and r["valor_flex"] and r["fecha_ok"])
    return r

def causa(r):
    if r["correcta"]:
        return "correcta"
    if not r["termino"]:
        return "no_termino"
    if not r["serie_ok"]:
        return "serie_equivocada"
    if not r["valor_ok"]:
        return "convencion_porcentaje" if r["valor_flex"] else "valor_equivocado"
    return "fecha_equivocada"

# ---------- Agente ----------
def generar(mensajes):
    ids = tokenizer.apply_chat_template(mensajes, tools=tools, add_generation_prompt=True,
                                        return_tensors="pt").to("cuda")
    n_in = ids.shape[1]
    if n_in > model.max_seq_length - MAX_NUEVOS:
        return None, n_in, 0, 0.0
    t = time.time()
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=MAX_NUEVOS, do_sample=False)
    nuevos = out[0][n_in:]
    return tokenizer.decode(nuevos, skip_special_tokens=False), n_in, len(nuevos), time.time() - t

RE_LLAMADA = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.S)

def episodio(pregunta, sistema):
    msgs = ([{"role": "system", "content": sistema}] if sistema else []) + [{"role": "user", "content": pregunta}]
    log = {"pasos": 0, "n_search": 0, "n_get": 0, "formato_invalido": False, "sin_llamada": False,
           "contexto_excedido": False, "submit": None, "ids_ultima": None, "ultimo_texto": ""}
    for paso in range(1, MAX_PASOS + 1):
        log["pasos"] = paso
        with lf.start_as_current_observation(as_type="generation", name=f"generacion-{paso}",
                                             model=ETIQUETA_MODELO, input=msgs[-1]) as gen:
            txt, n_in, n_out, seg = generar(msgs)
            gen.update(output=txt, metadata={"tokens_entrada": n_in, "tokens_salida": n_out, "segundos": round(seg, 2)})
        if txt is None:
            log["contexto_excedido"] = True
            break
        txt = txt.replace("<|im_end|>", "").strip()
        log["ultimo_texto"] = txt
        m = RE_LLAMADA.search(txt)
        if not m:
            log["sin_llamada"] = True
            break
        try:
            llamada = json.loads(m.group(1))
            nombre, args = llamada["name"], llamada["arguments"]
            assert isinstance(args, dict)
        except Exception:
            log["formato_invalido"] = True
            break
        msgs.append({"role": "assistant", "content": "",
                     "tool_calls": [{"id": f"call_{paso}", "type": "function",
                                     "function": {"name": nombre, "arguments": args}}]})
        if nombre == "submit_answer":
            log["submit"] = args
            break
        with lf.start_as_current_observation(as_type="span", name=f"herramienta:{nombre}", input=args) as ts:
            if nombre == "search_series":
                log["n_search"] += 1
                q = args.get("query")
                res = H.search_series(q) if isinstance(q, str) else json.dumps({"status": 400, "error": "query debe ser texto"})
            elif nombre == "get_series":
                log["n_get"] += 1
                log["ids_ultima"] = str(args.get("ids"))
                res = H.get_series(args)
            else:
                res = json.dumps({"status": 400, "error": f"herramienta desconocida: {nombre}"}, ensure_ascii=False)
            ts.update(output=res[:1500])
        msgs.append({"role": "tool", "tool_call_id": f"call_{paso}", "name": nombre, "content": res})
    return log, msgs

# ---------- Preguntas ----------
Fv = pd.read_csv(f"{RUTA}/eval_{SPLIT}_final2.csv")
Fv = Fv[Fv["marca_var"].fillna("") != "var_sobre_tasa"]
if ESTILO == "coloquial":
    Fv = Fv[~Fv["coloquial_ambigua"]]
S = pd.read_csv(f"{RUTA}/specs_{SPLIT}_v2.csv")
S["idx"] = S.index
Fv["idx"] = Fv["spec_id"].str.split("-").str[1].astype(int)
Fv = Fv.merge(S[["idx", "data"]], on="idx", how="left")
if N_PRUEBA:
    Fv = Fv.sample(frac=1, random_state=4).groupby("tipo").head(int(np.ceil(N_PRUEBA / 4)))
Fv = Fv.reset_index(drop=True)
if PROMPTS == ["basico"] and N_MUESTRA_BASICO:
    Fv = Fv.sample(n=min(N_MUESTRA_BASICO, len(Fv)), random_state=4).reset_index(drop=True)
print(f"Preguntas: {len(Fv)} ({SPLIT}, estilo {ESTILO}) | por tipo: {Fv['tipo'].value_counts().to_dict()} | prompts: {PROMPTS}")

def compacto(msgs):
    for m in msgs:
        if m["role"] == "system":
            print("  [sistema] (prompt optimizado)")
        elif m["role"] == "user":
            print("  USUARIO:", m["content"][:150])
        elif m.get("tool_calls"):
            f = m["tool_calls"][0]["function"]
            print("  LLAMADA:", f["name"], json.dumps(f["arguments"], ensure_ascii=False)[:230])
        else:
            print("  RESULTADO:", m["content"][:170])

resultados, trazas = [], []
SCORE_ERR, SCORE_OK = None, 0
t_total = time.time()
for nombre_prompt in PROMPTS:
    t_prompt, fc0 = time.time(), H.fallos_cache
    for r in Fv.to_dict("records"):
        params, d_gold = json.loads(r["params"]), json.loads(r["data"])
        gv, gf = valor_fecha(r["tipo"], r["frecuencia"], d_gold, params)
        g = {"valor": gv, "fecha": gf}
        pregunta = r[ESTILO]
        t0 = time.time()
        with lf.start_as_current_observation(
                as_type="span", name="pregunta", input={"pregunta": pregunta},
                metadata={"modelo": ETIQUETA_MODELO, "prompt": nombre_prompt, "tipo": r["tipo"],
                          "estilo": ESTILO, "split": SPLIT, "spec_id": r["spec_id"]}) as span:
            tid = lf.get_current_trace_id()
            log, msgs = episodio(pregunta, SISTEMA[nombre_prompt])
            sc = puntuar(r["tipo"], r["frecuencia"], g, log["submit"], log["ids_ultima"], params["ids"])
            span.update(output={"submit": log["submit"], "correcta": sc["correcta"], "causa": causa(sc)})
            try:
                span.score_trace(name="correcta", value=float(sc["correcta"]))
                SCORE_OK += 1
            except Exception as e:
                SCORE_ERR = f"{type(e).__name__}: {str(e)[:100]}"
        resultados.append({"prompt": nombre_prompt, "spec_id": r["spec_id"], "tipo": r["tipo"],
                           "frecuencia": r["frecuencia"], **sc, "causa": causa(sc),
                           **{k: log[k] for k in ["pasos", "n_search", "n_get", "formato_invalido",
                                                  "sin_llamada", "contexto_excedido"]},
                           "segundos": round(time.time() - t0, 1), "trace_id": tid})
        trazas.append({"prompt": nombre_prompt, "spec_id": r["spec_id"], "tipo": r["tipo"],
                       "pregunta": pregunta, "messages": msgs, "log": log, "score": sc})
        if len(resultados) % 25 == 0:
            pd.DataFrame(resultados).to_csv(f"{RUTA}/finetuned_{SPLIT}_{ESTILO}_{ETIQUETA}_parcial.csv", index=False)
            H.cache.commit()
            print(f"  [{len(resultados)}/{len(Fv) * len(PROMPTS)}] guardado parcial | {time.time() - t_total:.0f}s", flush=True)
    print(f"\n[{nombre_prompt}] terminado en {time.time() - t_prompt:.0f}s | fallos de caché (consultas nuevas a la API): {H.fallos_cache - fc0}")

lf.flush()
R = pd.DataFrame(resultados)

# ---------- Resumen ----------
print("\n" + "=" * 70)
for nombre_prompt in PROMPTS:
    D = R[R["prompt"] == nombre_prompt]
    print(f"\n### Prompt {nombre_prompt} (n = {len(D)})")
    print(f"correcta: {D['correcta'].mean():.0%} | correcta (acepta fracción): {D['correcta_flex'].mean():.0%} | "
          f"terminó con submit_answer: {D['termino'].mean():.0%} | serie correcta: {D['serie_ok'].mean():.0%}")
    print("por tipo:", (D.groupby("tipo")["correcta"].mean() * 100).round(0).to_dict())
    print("causas:", D["causa"].value_counts().to_dict())
    print(f"pasos (prom.): {D['pasos'].mean():.1f} | JSON inválido: {int(D['formato_invalido'].sum())} | "
          f"sin llamada: {int(D['sin_llamada'].sum())} | contexto excedido: {int(D['contexto_excedido'].sum())} | "
          f"segundos por pregunta: {D['segundos'].mean():.1f}")

print("\n" + "=" * 70 + "\nEjemplos de episodios (compactos)")
for nombre_prompt in PROMPTS:
    for tipo in ["variacion_interanual", "sin_datos"]:
        t = next((x for x in trazas if x["prompt"] == nombre_prompt and x["tipo"] == tipo), None)
        if t:
            print(f"\n--- [{nombre_prompt}] {tipo} | causa: {causa(t['score'])}")
            compacto(t["messages"])
            if t["log"]["sin_llamada"]:
                print("  (sin llamada) texto del modelo:", t["log"]["ultimo_texto"][:200])

print("\nScores en Langfuse:", "ok" if SCORE_ERR is None else f"NO funcionó ({SCORE_ERR})")
for t in R["trace_id"].head(2):
    try:
        print("URL de traza:", lf.get_trace_url(trace_id=t))
    except Exception as e:
        print("No pude armar la URL:", type(e).__name__)

# ---------- Guardado ----------
R.to_csv(f"{RUTA}/finetuned_{SPLIT}_{ESTILO}_{ETIQUETA}.csv", index=False)
with open(f"{RUTA}/finetuned_{SPLIT}_{ESTILO}_{ETIQUETA}_episodios.jsonl", "w") as f:
    for t in trazas:
        f.write(json.dumps(t, ensure_ascii=False, default=str) + "\n")
if SAVE_CACHE:
    H.cache.commit()
    shutil.copy("/content/cache_api.db", f"{RUTA}/cache_api.db")
    print("Caché actualizada en Drive")
print("Guardado:", f"finetuned_{SPLIT}_{ESTILO}_{ETIQUETA}.csv y _episodios.jsonl")
print(f"Tiempo total: {(time.time() - t_total) / 60:.0f} min")

In [ ]:
print("CELDA: COMPARACIÓN BASE VS. ENTRENADO + ANÁLISIS DE LA SELECCIÓN (SIN GPU, SIN COSTO)")
import json
import numpy as np
import pandas as pd
from scipy.stats import binomtest
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
AB = "baseline_val_tecnica_completa_optimizado"
FB = "finetuned_val_tecnica_completa_basico"
A, B = pd.read_csv(f"{RUTA}/{AB}.csv"), pd.read_csv(f"{RUTA}/{FB}.csv")
assert set(A["spec_id"]) == set(B["spec_id"]), "No son las mismas preguntas"
M = A.merge(B, on="spec_id", suffixes=("_base", "_ft"))

def wilson(k, n, z=1.96):
    p = k / n
    c = (p + z * z / (2 * n)) / (1 + z * z / n)
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return c - h, c + h

n = len(M)
print(f"Preguntas comparadas: {n}")
for nombre, col in [("Base + prompt optimizado", "correcta_base"), ("Entrenado, sin prompt", "correcta_ft")]:
    k = int(M[col].sum()); lo, hi = wilson(k, n)
    print(f"  {nombre}: {k}/{n} = {k / n:.1%} (IC 95%: {lo:.0%} a {hi:.0%})")

solo_base = int((M["correcta_base"] & ~M["correcta_ft"]).sum())
solo_ft = int((~M["correcta_base"] & M["correcta_ft"]).sum())
print(f"\nEmparejado: ambas {int((M['correcta_base'] & M['correcta_ft']).sum())} | solo el entrenado {solo_ft} "
      f"| solo la base {solo_base} | ninguna {int((~M['correcta_base'] & ~M['correcta_ft']).sum())}")
if solo_base + solo_ft:
    print(f"Prueba exacta sobre las {solo_base + solo_ft} donde difieren: p = "
          f"{binomtest(solo_ft, solo_base + solo_ft, 0.5).pvalue:.2g} "
          "(optimista: las preguntas salen de pocas familias y no son independientes)")

t = pd.DataFrame({"n": M.groupby("tipo_base").size(),
                  "base": M.groupby("tipo_base")["correcta_base"].mean() * 100,
                  "entrenado": M.groupby("tipo_base")["correcta_ft"].mean() * 100}).round(0)
t["diferencia"] = t["entrenado"] - t["base"]
print("\nPor tipo (% correcta):\n" + t.to_string())

# ---------- Análisis de la selección de serie ----------
S = pd.read_csv(f"{RUTA}/specs_val_v2.csv")
objetivo = lambda sid: json.loads(S.iloc[int(sid.split("-")[1])]["params"])["ids"]

def analizar(nombre):
    filas = []
    for l in open(f"{RUTA}/{nombre}_episodios.jsonl"):
        e = json.loads(l)
        obj, cand = objetivo(e["spec_id"]), None
        for m in e["messages"]:
            if m["role"] == "tool" and m.get("name") == "search_series":
                try:
                    cand = [c["serie_id"] for c in json.loads(m["content"])]
                except Exception:
                    pass
        elegida = e["log"].get("ids_ultima")
        en = bool(cand) and obj in cand
        filas.append({"tipo": e["tipo"], "obj_en_cand": en,
                      "rango_obj": cand.index(obj) + 1 if en else np.nan,
                      "eligio_primero": bool(cand) and elegida == cand[0],
                      "eligio_obj": elegida == obj})
    return pd.DataFrame(filas)

print("\nSelección de serie (último search_series de cada episodio):")
for nombre, etiqueta in [(AB, "Base + prompt optimizado"), (FB, "Entrenado, sin prompt")]:
    D = analizar(nombre)
    dentro, dificil = D[D["obj_en_cand"]], D[D["obj_en_cand"] & (D["rango_obj"] > 1)]
    print(f"\n  {etiqueta} (n = {len(D)})")
    print(f"    la serie correcta estaba entre los candidatos: {D['obj_en_cand'].mean():.0%}")
    print(f"    ...y la rango 1 era la correcta: {(dentro['rango_obj'] == 1).mean():.0%} de esos casos")
    print(f"    cuando estaba entre los candidatos, eligió la correcta: {dentro['eligio_obj'].mean():.0%}")
    print(f"    eligió el candidato 1: {D['eligio_primero'].mean():.0%}")
    print(f"    caso difícil (correcta NO en rango 1, n = {len(dificil)}): eligió la correcta "
          f"{dificil['eligio_obj'].mean():.0%} | eligió el candidato 1 {dificil['eligio_primero'].mean():.0%}")

In [ ]:
print("CELDA: DIAGNÓSTICO DE LAS CONSULTAS DE BÚSQUEDA (SIN GPU, SIN COSTO)")
import json, re, unicodedata
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
AB = "baseline_val_tecnica_completa_optimizado"
FB = "finetuned_val_tecnica_completa_basico"
S = pd.read_csv(f"{RUTA}/specs_val_v2.csv")
cat = (pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False)
         .drop_duplicates("serie_id").set_index("serie_id"))

def norm(s):
    s = unicodedata.normalize("NFKD", str(s).lower()).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s).strip()
def palabras(s):
    return set(re.findall(r"[a-z0-9]+", norm(s)))

def cargar(nombre):
    out = {}
    for l in open(f"{RUTA}/{nombre}_episodios.jsonl"):
        e = json.loads(l)
        obj = json.loads(S.iloc[int(e["spec_id"].split("-")[1])]["params"])["ids"]
        qs, cand = [], None
        for m in e["messages"]:
            if m.get("tool_calls"):
                f = m["tool_calls"][0]["function"]
                if f["name"] == "search_series":
                    qs.append(str(f["arguments"].get("query", "")))
            elif m["role"] == "tool" and m.get("name") == "search_series":
                try:
                    cand = [c["serie_id"] for c in json.loads(m["content"])]
                except Exception:
                    pass
        out[e["spec_id"]] = {"tipo": e["tipo"], "pregunta": e["pregunta"], "obj": obj, "queries": qs,
                             "en_cand": bool(cand) and obj in cand}
    return out

base, ft = cargar(AB), cargar(FB)

def resumen(nombre, d):
    q1 = [(v["queries"][0], v["pregunta"]) for v in d.values() if v["queries"]]
    largo = np.mean([len(q.split()) for q, _ in q1])
    digitos = np.mean([bool(re.search(r"\d", q)) for q, _ in q1])
    copia = np.mean([len(palabras(q) & palabras(p)) / max(1, len(palabras(q))) for q, p in q1])
    reform = np.mean([len(v["queries"]) > 1 for v in d.values()])
    print(f"\n{nombre} (n = {len(d)})")
    print(f"  palabras de la primera consulta: {largo:.1f} | con dígitos o años: {digitos:.0%} | "
          f"palabras de la consulta que están en la pregunta: {copia:.0%} | reformuló (más de una búsqueda): {reform:.0%}")
    t = pd.DataFrame([{"tipo": v["tipo"], "en_cand": v["en_cand"]} for v in d.values()])
    print("  serie correcta entre los candidatos, por tipo:",
          (t.groupby("tipo")["en_cand"].mean() * 100).round(0).to_dict())

resumen("Base + prompt optimizado", base)
resumen("Entrenado, sin prompt", ft)

perdidas = [k for k in ft if base[k]["en_cand"] and not ft[k]["en_cand"]]
ganadas = [k for k in ft if ft[k]["en_cand"] and not base[k]["en_cand"]]
print(f"\nLa base encontró la serie y el entrenado no: {len(perdidas)} | al revés: {len(ganadas)}")
rng = np.random.RandomState(1)
for k in rng.choice(perdidas, size=min(8, len(perdidas)), replace=False):
    print(f"\n  PREGUNTA: {ft[k]['pregunta'][:150]}")
    print(f"  consulta base     : {base[k]['queries'][:2]}")
    print(f"  consulta entrenado: {ft[k]['queries'][:2]}")
    print(f"  serie objetivo    : {cat.loc[ft[k]['obj'], 'serie_titulo']}")

In [ ]:
print("CELDA: ¿LAS CONSULTAS MÁS LARGAS ENCUENTRAN MEJOR LA SERIE? (SIN GPU, SIN COSTO)")
import os, re, json, shutil, sqlite3, unicodedata
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
AB = "baseline_val_tecnica_completa_optimizado"
FB = "finetuned_val_tecnica_completa_basico"
if not os.path.exists("/content/busqueda.db"):
    shutil.copy(f"{RUTA}/busqueda.db", "/content/busqueda.db")
con = sqlite3.connect("/content/busqueda.db")
S = pd.read_csv(f"{RUTA}/specs_val_v2.csv")

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [r[0] for r in rows]

def rango(q, obj):
    ids = buscar(q, 10) if q else []
    return ids.index(obj) + 1 if obj in ids else np.nan

RUIDO = {"variacion", "interanual", "respecto", "anterior", "ano", "anos", "valor", "mensual", "trimestral",
         "anual", "ultimo", "ultima", "reciente", "dato", "expresado", "expresada", "millones", "pesos",
         "dolares", "miles", "segun", "serie", "frecuencia", "periodo", "medida", "medido", "trimestre",
         "mismo", "cuando", "datos", "total", "enero", "febrero", "marzo", "abril", "mayo", "junio", "julio",
         "agosto", "septiembre", "setiembre", "octubre", "noviembre", "diciembre", "primer", "segundo",
         "tercer", "cuarto", "corrientes", "constantes"}
def sin_ruido(p):
    toks = re.findall(r"[a-z0-9]+", norm(p))
    return " ".join(t for t in toks if t not in RUIDO and not t.isdigit())

def primeras(nombre):
    out = {}
    for l in open(f"{RUTA}/{nombre}_episodios.jsonl"):
        e = json.loads(l)
        q = None
        for m in e["messages"]:
            if m.get("tool_calls") and m["tool_calls"][0]["function"]["name"] == "search_series":
                q = str(m["tool_calls"][0]["function"]["arguments"].get("query", ""))
                break
        obj = json.loads(S.iloc[int(e["spec_id"].split("-")[1])]["params"])["ids"]
        out[e["spec_id"]] = {"tipo": e["tipo"], "pregunta": e["pregunta"], "q": q, "obj": obj}
    return out

base, ft = primeras(AB), primeras(FB)
filas = []
for k, v in ft.items():
    b, sr = base[k], sin_ruido(v["pregunta"])
    filas.append({"id": k, "tipo": v["tipo"],
                  "base": rango(b["q"], v["obj"]), "entrenado": rango(v["q"], v["obj"]),
                  "pregunta_completa": rango(v["pregunta"], v["obj"]), "pregunta_sin_ruido": rango(sr, v["obj"]),
                  "n_base": len((b["q"] or "").split()), "n_ft": len((v["q"] or "").split()),
                  "n_sin_ruido": len(sr.split())})
D = pd.DataFrame(filas)
est = ["base", "entrenado", "pregunta_completa", "pregunta_sin_ruido"]

print(f"\nPreguntas: {len(D)}")
print("Recall de la serie correcta entre los 10 candidatos (%), por estrategia de consulta:")
print(pd.DataFrame({"@10": {e: (D[e] <= 10).mean() * 100 for e in est},
                    "@1": {e: (D[e] <= 1).mean() * 100 for e in est},
                    "palabras": {"base": D["n_base"].mean(), "entrenado": D["n_ft"].mean(),
                                 "pregunta_completa": np.nan, "pregunta_sin_ruido": D["n_sin_ruido"].mean()}}).round(1).to_string())
print("\nRecall @10 por tipo (%):")
print(pd.DataFrame({e: D.groupby("tipo")[e].apply(lambda s: (s <= 10).mean() * 100) for e in est}).round(0).to_string())

perd = D[(D["base"] <= 10) & ~(D["entrenado"] <= 10)]
print(f"\nCasos donde la consulta del base encontró la serie y la del entrenado no: {len(perd)}")
for e in ["pregunta_completa", "pregunta_sin_ruido"]:
    print(f"  de esos, los que habría encontrado la estrategia '{e}': {int((perd[e] <= 10).sum())}")

Q = pd.concat([pd.DataFrame({"n": D["n_base"], "hit": D["base"] <= 10}),
               pd.DataFrame({"n": D["n_ft"], "hit": D["entrenado"] <= 10})])
Q["tramo"] = pd.cut(Q["n"], [0, 3, 4, 5, 6, 8, 100], labels=["3 o menos", "4", "5", "6", "7-8", "9 o más"])
print("\nRecall @10 según el largo de la consulta (las consultas de los dos modelos juntas):")
print(Q.groupby("tramo", observed=True)["hit"].agg(consultas="size", recall="mean").round(2).to_string())

P = pd.read_csv(f"{RUTA}/consultas_busqueda_train_parcial.csv")
P = P[P["ok"] == True]
n_tr = P["consulta"].astype(str).str.split().str.len()
print(f"\nConsultas de ENTRENAMIENTO (escritas por Haiku, n = {len(P)}): {n_tr.mean():.1f} palabras en promedio | "
      f"{(n_tr <= 5).mean():.0%} con 5 palabras o menos | {(n_tr >= 7).mean():.0%} con 7 o más")

In [ ]:
print("CELDA: REGLAS PARA ARMAR LA CONSULTA DE BÚSQUEDA, ELEGIDAS SOBRE TRAIN (SIN GPU, SIN COSTO)")
import os, re, json, shutil, sqlite3, unicodedata
import numpy as np
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
if not os.path.exists("/content/busqueda.db"):
    shutil.copy(f"{RUTA}/busqueda.db", "/content/busqueda.db")
con = sqlite3.connect("/content/busqueda.db")

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                       "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [r[0] for r in rows]

def rango(q, obj):
    ids = buscar(q, 10) if q else []
    return ids.index(obj) + 1 if obj in ids else np.nan

MESES = {"enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "setiembre", "octubre", "noviembre", "diciembre"}
ORD = {"primer", "segundo", "tercer", "cuarto"}
MULETILLAS = {"variacion", "interanual", "respecto", "anterior", "ano", "mismo", "ultimo", "ultima",
              "reciente", "dato", "cual", "fue", "cuanto"}

def toks(p):
    return re.findall(r"\w+", str(p))
def r_completa(p):
    return str(p)
def r_sin_digitos(p):
    return " ".join(t for t in toks(p) if not re.search(r"\d", t))
def r_sin_fechas(p):
    return " ".join(t for t in toks(p) if not re.search(r"\d", t) and norm(t) not in MESES and norm(t) not in ORD)
def r_sin_fechas_ni_muletillas(p):
    return " ".join(t for t in toks(p) if not re.search(r"\d", t) and norm(t) not in MESES
                    and norm(t) not in ORD and norm(t) not in MULETILLAS)
REGLAS = {"completa": r_completa, "sin_digitos": r_sin_digitos, "sin_fechas": r_sin_fechas,
          "sin_fechas_ni_muletillas": r_sin_fechas_ni_muletillas}

# ---------- Universo de TRAIN (el mismo que usaron las trayectorias) ----------
F = pd.read_csv(f"{RUTA}/train_final2.csv")
F = F[F["marca_var"].fillna("") != "var_sobre_tasa"]
filas = []
for r in F.to_dict("records"):
    filas.append({"qid": f"{r['spec_id']}-tecnica", "tipo": r["tipo"], "estilo": "tecnica",
                  "pregunta": r["tecnica"], "serie_id": r["serie_id"]})
    if not r["coloquial_ambigua"] and r["tipo"] != "sin_datos":
        filas.append({"qid": f"{r['spec_id']}-coloquial", "tipo": r["tipo"], "estilo": "coloquial",
                      "pregunta": r["coloquial"], "serie_id": r["serie_id"]})
Q = pd.DataFrame(filas)
print("Preguntas de entrenamiento:", len(Q), "| por estilo:", Q["estilo"].value_counts().to_dict())

P = pd.read_csv(f"{RUTA}/consultas_busqueda_train_parcial.csv")
P = P[P["ok"] == True]
haiku = dict(zip(P["qid"].astype(str), P["consulta"]))
Q["haiku_actual"] = [rango(haiku.get(q), s) for q, s in zip(Q["qid"], Q["serie_id"])]
for nombre, f in REGLAS.items():
    Q[nombre] = [rango(f(p), s) for p, s in zip(Q["pregunta"], Q["serie_id"])]

cols = ["haiku_actual"] + list(REGLAS)
print("\nTRAIN: serie correcta entre los 10 candidatos (@10) y como primer candidato (@1), en %:")
print(pd.DataFrame({"@10": {c: (Q[c] <= 10).mean() * 100 for c in cols},
                    "@1": {c: (Q[c] <= 1).mean() * 100 for c in cols}}).round(1).to_string())
print("\n@10 por estilo de pregunta (%):")
print(pd.DataFrame({c: Q.groupby("estilo")[c].apply(lambda s: (s <= 10).mean() * 100) for c in cols}).round(1).to_string())
print("\n@10 por tipo (%):")
print(pd.DataFrame({c: Q.groupby("tipo")[c].apply(lambda s: (s <= 10).mean() * 100) for c in cols}).round(1).to_string())
print("\nLargo de la consulta (palabras): promedio / máximo")
for nombre, f in REGLAS.items():
    n = Q["pregunta"].map(lambda p: len(f(p).split()))
    print(f"  {nombre}: {n.mean():.1f} / {n.max()}")
print("\nEjemplos (pregunta -> consulta según la regla 'sin_fechas'):")
for p in Q.sample(4, random_state=2)["pregunta"]:
    print("  -", str(p)[:110], "\n    ->", r_sin_fechas(p)[:110])

# ---------- Confirmación en VALIDACIÓN (no se usa para elegir) ----------
S = pd.read_csv(f"{RUTA}/specs_val_v2.csv")
V = pd.read_csv(f"{RUTA}/eval_val_final2.csv")
V = V[V["marca_var"].fillna("") != "var_sobre_tasa"]
V["obj"] = [json.loads(S.iloc[int(s.split("-")[1])]["params"])["ids"] for s in V["spec_id"]]
for nombre, f in REGLAS.items():
    V[nombre] = [rango(f(p), o) for p, o in zip(V["tecnica"], V["obj"])]
print(f"\nVALIDACIÓN (solo confirmación, n = {len(V)}): @10 y @1 en %")
print(pd.DataFrame({"@10": {c: (V[c] <= 10).mean() * 100 for c in REGLAS},
                    "@1": {c: (V[c] <= 1).mean() * 100 for c in REGLAS}}).round(1).to_string())

In [ ]:
!pip -q install transformers
print("CELDA: TRAYECTORIAS v2 (CONSULTA = PREGUNTA SIN FECHAS NI MULETILLAS) - CPU, SIN COSTO")
import os, re, json, shutil, sqlite3, unicodedata
from collections import Counter
import numpy as np
import pandas as pd
from google.colab import drive
from transformers import AutoTokenizer

drive.mount("/content/drive")
RUTA = "/content/drive/MyDrive/qlora-series-ar/data"
SALIDA = f"{RUTA}/trayectorias_train_v2.jsonl"
PCT_SIN_DATOS = 0.18

def norm(s):
    s = "" if pd.isna(s) else str(s)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", s.lower().replace("_", " ")).strip()

STOP = {"de", "del", "la", "el", "los", "las", "en", "y", "a", "por", "que", "cual",
        "fue", "cuanto", "es", "un", "una", "para", "con", "se", "al", "lo"}
if not os.path.exists("/content/busqueda.db"):
    shutil.copy(f"{RUTA}/busqueda.db", "/content/busqueda.db")
con = sqlite3.connect("/content/busqueda.db")

def buscar(q, k=10):
    toks = [t for t in re.findall(r"[a-z0-9]+", norm(q)) if t not in STOP and len(t) > 1]
    toks = list(dict.fromkeys(toks))
    if not toks:
        return []
    partes = [f'"{t[:5]}"*' if len(t) >= 6 else f'"{t}"' for t in toks]
    rows_ = con.execute("SELECT serie_id, bm25(fts, 0.0, 2.0, 1.0) AS s FROM fts "
                        "WHERE fts MATCH ? ORDER BY s LIMIT ?", (" OR ".join(partes), k)).fetchall()
    return [x[0] for x in rows_]

# ---------- La regla de consulta (elegida sobre train) ----------
MESES = {"enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto",
         "septiembre", "setiembre", "octubre", "noviembre", "diciembre"}
ORD = {"primer", "segundo", "tercer", "cuarto"}
MULETILLAS = {"variacion", "interanual", "respecto", "anterior", "ano", "mismo", "ultimo", "ultima",
              "reciente", "dato", "cual", "fue", "cuanto"}
def consulta_de(p):
    out, vistos = [], set()
    for t in re.findall(r"\w+", str(p)):
        n = norm(t)
        if (re.search(r"\d", t) or n in MESES or n in ORD or n in MULETILLAS
                or n in STOP or len(n) <= 1 or n in vistos):
            continue
        vistos.add(n)
        out.append(t)
    return " ".join(out)

# ---------- Mismas preguntas que en v1 ----------
tools = json.load(open(f"{RUTA}/tools_v1.json"))
E = pd.read_csv(f"{RUTA}/specs_train_v1.csv")
E["spec_id"] = E.index
F = pd.read_csv(f"{RUTA}/train_final2.csv")
F = F[F["marca_var"].fillna("") != "var_sobre_tasa"]
F = F.merge(E[["spec_id", "data", "respuesta_final", "serie_titulo", "serie_unidades"]], on="spec_id", how="left")
rows = []
for r in F.to_dict("records"):
    rows.append({**r, "estilo": "tecnica", "pregunta": r["tecnica"]})
    if not r["coloquial_ambigua"]:
        rows.append({**r, "estilo": "coloquial", "pregunta": r["coloquial"]})
Q = pd.DataFrame(rows)
non = Q[Q["tipo"] != "sin_datos"]
sin = Q[(Q["tipo"] == "sin_datos") & (Q["estilo"] == "tecnica")]
n_sin = int(round(PCT_SIN_DATOS / (1 - PCT_SIN_DATOS) * len(non)))
sin = sin.sample(n=min(n_sin, len(sin)), random_state=3)
Q = pd.concat([non, sin]).sample(frac=1, random_state=5).reset_index(drop=True)
Q["qid"] = Q["spec_id"].astype(str) + "-" + Q["estilo"]
print("Preguntas candidatas:", len(Q), "| por tipo:", Q["tipo"].value_counts().to_dict())

info = pd.read_csv(f"{RUTA}/catalogo_busqueda.csv", low_memory=False).drop_duplicates("serie_id").set_index("serie_id")

def periodo(fecha, freq):
    return fecha[:4] if freq == "anual" else fecha[:7]

def args_submit(r, d, params):
    if r["tipo"] == "sin_datos":
        txt = re.sub(r"\s*La serie tiene datos entre.*$", "", r["respuesta_final"])
        return {"respuesta": txt, "valor": None, "fecha": None}
    if r["tipo"] == "ultimo_dato":
        return {"respuesta": r["respuesta_final"], "valor": d[0][1], "fecha": d[0][0]}
    if r["tipo"] == "valor_puntual":
        return {"respuesta": r["respuesta_final"], "valor": d[0][1], "fecha": params["start_date"]}
    return {"respuesta": r["respuesta_final"], "valor": round(d[-1][1] * 100, 2),
            "fecha": periodo(d[-1][0], r["frecuencia"])}

def llamada(i, nombre, args):
    return {"role": "assistant", "content": "",
            "tool_calls": [{"id": f"call_{i}", "type": "function",
                            "function": {"name": nombre, "arguments": args}}]}

trayectorias, perdidas = [], Counter()
for r in Q.to_dict("records"):
    consulta = consulta_de(r["pregunta"]).strip()
    if not consulta:
        perdidas["consulta vacía"] += 1
        continue
    ids = buscar(consulta, 10)
    if r["serie_id"] not in ids:
        perdidas[f"la serie no aparece en el top 10 ({r['estilo']})"] += 1
        continue
    d, params = json.loads(r["data"]), json.loads(r["params"])
    cand = [{"serie_id": s, "titulo": str(info.loc[s, "serie_titulo"]),
             "distribucion": str(info.loc[s, "distribucion_titulo"]),
             "frecuencia": str(info.loc[s, "frecuencia"])} for s in ids]
    get_res = {"status": 200, "serie_id": r["serie_id"], "titulo": r["serie_titulo"],
               "unidades": None if pd.isna(r["serie_unidades"]) else r["serie_unidades"],
               "frecuencia": r["frecuencia"], "data": d}
    msgs = [{"role": "user", "content": r["pregunta"]},
            llamada(1, "search_series", {"query": consulta}),
            {"role": "tool", "tool_call_id": "call_1", "name": "search_series",
             "content": json.dumps(cand, ensure_ascii=False)},
            llamada(2, "get_series", params),
            {"role": "tool", "tool_call_id": "call_2", "name": "get_series",
             "content": json.dumps(get_res, ensure_ascii=False)},
            llamada(3, "submit_answer", args_submit(r, d, params))]
    trayectorias.append({"qid": r["qid"], "spec_id": int(r["spec_id"]), "tipo": r["tipo"],
                         "estilo": r["estilo"], "serie_id": r["serie_id"], "familia": r["familia"],
                         "rango_busqueda": ids.index(r["serie_id"]) + 1, "messages": msgs})

T = pd.DataFrame([{k: v for k, v in t.items() if k != "messages"} for t in trayectorias])
print(f"\nTrayectorias v2: {len(T)} de {len(Q)} | perdidas: {dict(perdidas)}")
print("Por tipo:", T["tipo"].value_counts().to_dict(), "| por estilo:", T["estilo"].value_counts().to_dict())
print(f"% sin_datos: {(T['tipo'] == 'sin_datos').mean():.1%} | series: {T['serie_id'].nunique()} | familias: {T['familia'].nunique()}")

# ---------- Comparación con v1 ----------
V1 = pd.DataFrame([{k: v for k, v in json.loads(l).items() if k != "messages"}
                   for l in open(f"{RUTA}/trayectorias_train_v1.jsonl")])
print(f"\nv1: {len(V1)} trayectorias | la serie correcta era el candidato 1 en {(V1['rango_busqueda'] == 1).mean():.0%} "
      f"| por estilo: {V1['estilo'].value_counts().to_dict()}")
print(f"v2: {len(T)} trayectorias | la serie correcta es el candidato 1 en {(T['rango_busqueda'] == 1).mean():.0%} "
      f"| rango mediano {int(T['rango_busqueda'].median())} | por estilo: {T['estilo'].value_counts().to_dict()}")
print("Largo de la consulta (palabras):", round(np.mean([len(t['messages'][1]['tool_calls'][0]['function']['arguments']['query'].split())
                                                       for t in trayectorias]), 1))
print("\nEjemplos (pregunta -> consulta -> rango):")
for t in trayectorias[:4]:
    print("  -", t["messages"][0]["content"][:100], "\n    ->", t["messages"][1]["tool_calls"][0]["function"]["arguments"]["query"][:100],
          "| rango", t["rango_busqueda"])

# ---------- Largo en tokens (tokenizer de Qwen, en CPU) ----------
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-4B-Instruct-2507")
largos = [len(tok(tok.apply_chat_template(t["messages"], tools=tools, tokenize=False),
                  add_special_tokens=False)["input_ids"]) for t in trayectorias]
print(f"\nTokens por trayectoria: media {np.mean(largos):.0f} | p99 {np.percentile(largos, 99):.0f} | máx {max(largos)} (límite 2048)")

with open(SALIDA, "w") as f:
    for t in trayectorias:
        f.write(json.dumps(t, ensure_ascii=False) + "\n")
print("\nGuardado:", SALIDA, "(no pisa trayectorias_train_v1.jsonl)")